# Methoden en Technieken 2025-2026 -- Blok 3
<h1 style="background-color:crimson">OLD</h1>

## Datapunt Opdracht 3a

In deze opdracht worden de volgende leeruitkomsten getoetst, relevante termen zijn **dik** gedrukt:
- A2: Je stelt voor een AI-oplossing juridische, ethische, organisatorische, **functionele en technische requirements** op. ✅
- B1: Je **verkent en prepareert een dataset voor het trainen en testen van een AI-model en kan de voor- en nadelen van het gebruik van een bestaande dataset onderbouwen**, rekening houdend met technische en ethische randvoorwaarden. ✅
- B2: Je **stelt op basis van requirements en data een geschikte architectuur voor een AI-oplossing op en selecteert daarvoor passende AI-technieken gebruik makend van bijvoorbeeld** **machine learning**, deep learning, kennisrepresentatie, computer vision en **natural language processing**. ✅
- B3: Je **ontwikkelt een nieuw** of voorgetraind **AI-model volgens een iteratief en systematisch proces**.
- C2: **Je evalueert en beoordeelt de kwaliteit van een AI-model aan de hand van kwaliteitscriteria die in het vakgebied erkend worden** zoals robustness, **performance**, scalability, explainability, **model complexity** en resource demand.


## De opdracht

Onderstaande code leest de data van verschillende *ratings* in. Deze dataset is de **MovieTweetings**-dataset (ook naar verwezen in Les 4 van blok 3) waar het MovieGEEKs-voorbeeld gebruik van maakt. In de data staan de waarderingen (van 0 t/m 10) van gebruikers voor verschillende films en bijbehorende *timestamp*. Zie ook https://github.com/sidooms/MovieTweetings/tree/master voor een uitleg van de dataset.

In [1]:
import pandas as pd
import surprise # Zie https://surpriselib.com/

In [2]:
ratings = pd.read_csv('https://raw.githubusercontent.com/sidooms/MovieTweetings/master/latest/ratings.dat',
                      delimiter='::', engine='python', header=None,
                      names = ['User_ID', 'Movie_ID', 'Rating', 'Rating_Timestamp'])

De bedoeling is om een aanbevelings-systeem te bouwen dat voor elke willekeurige gebruiker in het systeem drie films aanbeveelt. Probeer de aanbeveling zo persoonlijk mogelijk te maken.
* Kies een model en verantwoord deze keuze. ✅
* Besluit hoe je het model beoordeelt (datasplitsing en maatstaf) en verantwoord deze keuze. ✅
* Evalueer het model. 
* Geef concrete suggesties om het model te verbeteren. Je hoeft deze verbeteringen niet uit te voeren.
* Bespreek voor- en nadelen van het model dat je hebt gemaakt.

----

# 1. Modelkeuze - (Kies een model en verantwoord deze keuze)

## 1.1. Algoritme
Kim falk bespreekt dat er 3 soorten algortimes zijn binnen recommender systems: Collaborative Filtering, Content-Based Filtering en Hybrid Recommender. Ik heb geen meta-data voor de content, dus dan vallen de approaches content-based en hybrid al af. Hierdoor kan ik alleen collaborative filtering toepassen wat ik dus ook ga doen.

## 1.2. Model
... (kies een model) zoek in het boek de modellen voor hybrid

# 2. Datasplitsing en Maatstaf - (Besluit hoe je het model beoordeelt (datasplitsing en maatstaf) en verantwoord deze keuze.)

In [ ]:
# code voor data splitsing enz naar hier verplaatsen

## 2.1. Datasplitsing
Falk zegt: "It’s a good idea to split the data around 80–20, but that isn’t a rule.  It’s important to have as much data as possible to train the algorithm," ook zegt hij eerder dat het goed is om een train test en validatieset te gebruiken. Daarom ga ik de sets spltisen op de volgende manier: train (80%) test (10%) en validatie set (10%). 

## 2.2. Crossvalidatie
Crossvalidatie zou in dit scenario een goede keus zijn omdat de validatie set erg fragiel is. Wat ik daarmee bedoel is dat de set klein is, en het kleine beetje data dat in de validatieset zit kan de precision score enorm beïnvloeden. Je zou bijna zeggen dat je met de validatieset een beetje "geluk" moet hebben want wat is de kans dat we daadwerkelijk een film aanraden die de user in de validatieset al gezien heeft. Dit kunnen we dus oplossen met cross validatie om een meer zekere score te krijgen en randomness bias in de validatieset wet ge werken. ECHTER, duurt het trainen van de top_n_recomendaties zo ontzettend lang dat ik het voor de opdracht i.v.m tijd liever niet doe, maar als tijd geen kwestie was zou ik het zeker meenemen in deze opdracht.

## 2.3. Baseline Model
Falk zegt: "Before you evaluate your new recommender, you should evaluate it on a simple recommender that, for example, always recommends the most popular items and see what numbers come out." Michelangelo gaf in de les ook aan dat het een goed idee is om een baseline te gebruiken en het recommenden van populaire items daar een goed idee voor is. Daarom ga ik voor mijn baseline populair films aanraden. *Maar wat is populair?* de best beoordeelde films gebruiken kan zorgen voor problemen, aangezien een film die 2 keer beoordeeld is met een gemmiddelde score van 10.0 hoger kan staan dan een film die 1000 keer is beoordeeld met een gemmiddelde score van 9.9. Om dit op te lossen ga ik populariteit definiëren als vaakst beoordeeld.

## 2.4. Evaluatie Metriek
Ten eerste is het best awkward om een offline experiment uit te voeren voor een recommender system aangezien je in feiten goeie recomendations zou kunnen maken, maar je krijgt nooit feedback terug van de user. Falk noemt meerdere evaluatie metrics zoals: RMSE, MAE, DCG, NDCG, Precision, Recall en F1-Score. Ik ben van mening dat metrics als RMSE, MAE, DCG, NDCG goed zouden kunnen werken in de praktijk. je recommend een film, wacht af tot de gebruiker hem kijkt of niet, eventueel hoeveel sterren de user de film beoordeeld en kunt dat gebruiken als feedback om verder te modelleren. Maar omdat we dus statische data hebben moeten we het doen met wat we hebben. Daarom lijkt mij het meest logisch om een rating te interpreteren als "heeft film gezien" en geen rating als "heeft film niet gezien". Zo kunnen we daadwerkelijk meten hoe goed het model presteert door te proberen films aan te raden die de user ook daadwerkelijk gezien heeft. Dit sluit mooi aan op de metrieken precision, recall en f1-score aangezien het nu meer een classificatie probleem wordt. Aangezien de opdracht is om enkel 3 films aan te bevelen is het onlogisch om te kijken naar recall, aangezien je 3 van de 3 films goed kunt aanbevelen, maar als de gebruiker in totaal 30 films heeft beoordeeld dan heb je dus maar een recall van 0.10. omdat recall onlogisch is valt f1-score ook af dus is de meest logische metriek in mijn opvatting van deze opdracht precision.

Nogmaals, ik ben zelf niet fan van het recommender systeem omzetten in een classificatie probleem, maar omdat we geen livefeedback krijgen van users lijkt mij dit de meest logische oplossing.

_het nadeel hiervan is dat we het goed rekenen als user een film gezien heeft eigenlijk moet je dit splitsen in gezien goed / gezien slecht / niet gezien. heeft film wel gezien kan ook betekenen dat die slecht was, heeft film niet gezien kan ook betekenen dat die hem nog niet ontdekt heeft._

# 3. Baseline model implementatie
Voor mijn baseline model ga ik dus in de train set de 3 meest beoordeelde films opzoeken en deze aan iedereen aanraden op de validatie set. Vervolgens reken ik de precision uit en kan ik zo weten hoe goed het model scoort.

drop timestamps

In [3]:
ratings.drop(columns=['Rating_Timestamp'], inplace=True) # ik gebruik geen timestamps

movies sorteren op meest beoordeeld tot minst beoordeeld

In [4]:
from sklearn.model_selection import train_test_split

# split 80/10/10
train_data, temp_data = train_test_split(ratings, test_size=0.2, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

# meest populair films zoeken
movie_stats = train_data.groupby('Movie_ID')['Rating'].agg(['mean', 'count'])
movie_stats = movie_stats.sort_values(by='count', ascending=False)
movie_stats

,mean,count
Movie_ID,,
1454468,8.212010,2448
816692,8.823480,2368
8579674,8.625853,2344
993846,8.412788,2299
7286456,9.068548,2232
...,...,...
1073654,7.000000,1
1073096,10.000000,1
1071211,7.000000,1


top 3 meest beoordeelde Movie_IDs

In [5]:
top_3_popular_ids = movie_stats.sort_values(by='count', ascending=False).head(3).index.tolist()
top_3_popular_ids


[1454468, 816692, 8579674]

validatieset restructuren zodat per user elk beoordeelde movie op een rij staat

In [6]:
user_val_movies = val_data.groupby('User_ID')['Movie_ID'].apply(set).reset_index()
user_val_movies.columns = ['User_ID', 'actual_movies']
user_val_movies

,User_ID,actual_movies
0,1,{114508}
1,3,"{8946378, 7975244, 6751668, 1568346, 118715}"
2,9,"{3076658, 810819, 1392190}"
3,15,{181875}
4,27,{5867314}
...,...,...
24941,71694,"{6802308, 1860357, 1396484, 1959563, 88847, 80..."
24942,71700,"{443473, 805570}"
24943,71702,"{2474976, 1457765, 2361509, 4972582, 2709768, ..."
24944,71704,{1259521}


Precision uitrekenen

In [7]:
def calculate_precision(actual, predicted):
    # Find intersection
    hits = len(set(predicted) & set(actual))
    return hits / len(predicted)

# Apply calculation to every user
user_val_movies['precision'] = user_val_movies['actual_movies'].apply(
    lambda x: calculate_precision(x, top_3_popular_ids)
)

# Final Average Precision
avg_precision = user_val_movies['precision'].mean()

print(f"Average Precision: {avg_precision:.4f} ({avg_precision:.2%})")

Average Precision: 0.0125 (1.25%)


Dit houdt dus in dat het baseline model in 1.25% van de gevallen een film heeft aangeraden die de user ook daadwerkelijk heeft gezien. Dat betekent dat de user op een bepaald punt in tijd interesse heeft getoond in deze film.

# 4. Functionele en Technische Requirements (A2)
- R1. **Precision als evaluatie metriek** - Ik heb geprobeerd uit te leggen waarom ik van mening ben dat in dit fictief scenario precision een goede keus is. De hoofdreden hiervoor is omdat we in geen enkel moment in tijd feedback kunnen krijgen van users. We hebben een statische dataset en daarom ben ik van mening dat het het meest logisch is om voor dit fictief scenario precision te gebruiken als metric. Voor uitgebreidere beredenering zie [*2.3. Evaluatie Metriek*](#24-evaluatie-metriek).
- R2. **Het model moet een precision score van > 0.0125 behalen** - (gebasseerd op resultaten van het baseline model)
- R3. **Collaborative filtering gebruiken als algortime** - Er is geen movie meta-data in de gegeven dataset waardoor andere approaches niet mogelijk zijn.
- R4. **3 films aanraden per user** - staat in de opdracht.
- R5. **Minimaal 1 iteratie inclusief verbeter punten voor een volgende iteratie** - tot deze conclusie gekomen uit de vragen in de les (nulmodel telt natuurlijk niet als losse iteratie)
- R6. **80/10/10 data split** - aanrader van K. Falk (voor uitgebreidere verantwoording zie [*2.1. Datasplitsing*](#21-datasplitsing))
- R7. **... Model Gebruiken** - ?
- R8. **... Cross-Validatie** - ?


# 5. Model Implementatie
Hier ga ik dus een model bouwen dat met matrix factorization gaat predicten wat de meest interessante films zijn voor een user. Ik ga de films aanraden die de matrix factorization het hoogst predict, mochten er 2 films zijn met de zelfde geschatte rating maar een van de twee films is vaker beoordeeld dan wordt die voorgetrokken.

Ten slotte ga ik met mijn validatie set evalueren of dat de predicted 3 movies ook daadwerkelijk staan in de validatieset en zo de precision uitrekenen.

In [ ]:
from surprise import SVD
from surprise import Dataset, Reader

reader = Reader(rating_scale=(1, 10))
surprise_data = Dataset.load_from_df(train_data[['User_ID', 'Movie_ID', 'Rating']], reader)

trainset = surprise_data.build_full_trainset()

svd = SVD(n_factors=100, biased=False)  # 100 because tutorial on redis.io did so too, kim falk explains you should experiment with A/B testing what the best number is for latent dimensions but not possible time constraint
svd.fit(trainset) # Pass 'trainset', not 'surprise_data'

In [16]:
user_vectors = svd.pu  # user latent features (matrix)
movie_vectors = svd.qi  # movie latent features (matrix)

print(f'we have {user_vectors.shape[0]} users with feature vectors of size {user_vectors.shape[1]}')
print(f'we have {movie_vectors.shape[0]} movies with feature vectors of size {movie_vectors.shape[1]}')


we have 65021 users with feature vectors of size 100
we have 34694 movies with feature vectors of size 100


predicting rating for a single user

In [ ]:
import numpy as np

# surprise casts userId and movieId to inner ids, so we have to use their mapping to know which rows to use
inner_uid = trainset.to_inner_uid(347) # userId
inner_iid = trainset.to_inner_iid(20641) # movieId

# predict one user's rating of one film
predicted_rating = np.dot(user_vectors[inner_uid], movie_vectors[inner_iid])
print(f'the predicted rating of user {347} on movie {5515} is {predicted_rating}')

the predicted rating of user 347 on movie 5515 is 1.8117727824629317


kolommen in movie stats zitten niet op een lijn om een of andere reden dus ff dat fixen

In [ ]:
# Flatten the hierarchy and bring Movie_ID back as a normal column
movie_stats.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in movie_stats.columns.values]
movie_stats = movie_stats.reset_index()

# Create dictionary
movie_popularity_map = movie_stats.set_index('Movie_ID')['count'].to_dict()

ValueError: cannot insert level_0, already exists

top 3 for alle users uitrekenen

In [32]:
def get_top_n_recommendations(algo, trainset, popularity_map, n=3):
    top_n = defaultdict(list)
    
    # Pre-fetch all raw movie IDs once to save time
    all_movie_raw_ids = [trainset.to_raw_iid(i) for i in trainset.all_items()]
    
    for user_inner_id in trainset.all_users():
        user_raw_id = trainset.to_raw_uid(user_inner_id)
        
        # Get movies this user has already seen (internal IDs)
        user_items = {item_inner_id for (item_inner_id, _) in trainset.ur[user_inner_id]}
        
        predictions = []
        for movie_raw_id in all_movie_raw_ids:
            # Check if user already watched it
            if trainset.to_inner_iid(movie_raw_id) in user_items:
                continue
            
            # Predict rating
            pred = algo.predict(user_raw_id, movie_raw_id).est
            
            # Get count (default to 0 if movie isn't in our stats)
            count = popularity_map.get(movie_raw_id, 0)
            
            predictions.append((movie_raw_id, pred, count))
        
        # TIE-BREAKER LOGIC: 
        # Sort by predicted score first (index 1), then by count (index 2)
        predictions.sort(key=lambda x: (x[1], x[2]), reverse=True)
        
        top_n[user_raw_id] = [movie_id for (movie_id, score, count) in predictions[:n]]
        
    return top_n

# Now run it with the map included
recommendations = get_top_n_recommendations(svd, trainset, movie_popularity_map, n=3)

de top 3 movie ids per user

In [33]:
recommendations

defaultdict(list,
            {42431: [1302006, 2125608, 109830],
             1633: [8579674, 1371111, 81505],
             64157: [816692, 11337862, 8579674],
             40861: [816692, 1483013, 2179136],
             15932: [7286456, 4550098, 1255953],
             52921: [8579674, 7286456, 109830],
             14948: [1343092, 1843866, 4154796],
             66182: [2084970, 1798709, 2975590],
             21084: [7286456, 54953, 253474],
             20984: [7286456, 453562, 816692],
             25825: [54953, 6966692, 109830],
             25144: [7286456, 11032374, 111161],
             45496: [5726616, 62622, 6723592],
             46307: [54953, 7286456, 68646],
             35563: [7286456, 1392214, 8946378],
             41836: [3783958, 2119532, 1895587],
             65864: [1454468, 816692, 8579674],
             1943: [109830, 68646, 1160419],
             20327: [7286456, 2119532, 2488496],
             43382: [7286456, 482571, 86879],
             30237: [8579674, 

evaluatie met validatieset

In [35]:
def calculate_precision_at_3(recommendations, validation_df):
    # 1. Group the validation set: {User_ID: set(Movies_they_actually_watched)}
    # We use a set for O(1) lightning-fast lookups
    actual_watched = validation_df.groupby('User_ID')['Movie_ID'].apply(set).to_dict()
    
    precisions = []

    for user_id, rec_list in recommendations.items():
        # Only evaluate users who actually appear in our validation set
        if user_id in actual_watched:
            user_actual = actual_watched[user_id]
            
            # 2. Find the intersection (the "Hits")
            # How many of our Top 3 were actually in the validation set?
            hits = len(set(rec_list) & user_actual)
            
            # 3. Precision = Hits / Total Recommended (3)
            precision = hits / len(rec_list) if len(rec_list) > 0 else 0
            precisions.append(precision)
            
    # Return the average across all users
    return sum(precisions) / len(precisions) if precisions else 0

# Run the evaluation
# Run the evaluation
avg_precision = calculate_precision_at_3(recommendations, val_data)

# Print both formats
print(f"Average Precision @ 3 (Decimal): {avg_precision:.4f}")
print(f"Average Precision @ 3 (Percent): {avg_precision:.2%}")

Average Precision @ 3 (Decimal): 0.0091
Average Precision @ 3 (Percent): 0.91%


best wel sneu dit

## Evalueer het model
... figuren tabellen grafieken iets waardoor iemand kan lezen wat er gebeurt zonder domein info

## Suggesties voor Verbeteren
... hm tijd integreren die in de dataset staat. ook kunn je een andere dataset gaan zoeken waar wel content meta data instaat en een hybrid model bouwen.

## Voor- en Nadelen van het model
... wat zijn de voor en nadelen van CF

# check nog ff of alle bronnen kloppen en verwijzingen